## Expected Credit Loss

### Importing Libraries

In [1]:
import numpy as np
import pandas as pd

import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.append(str(project_root))

### Loading Dataset

In [2]:
from src.preprocessing import (
    load_data,
    rename_columns,
    convert_date_columns,
    replace_never_delinquent_value,
)


In [3]:
data_path = "../data/raw/dataset"

loan_data = load_data(data_path)

loan_data = rename_columns(loan_data)
loan_data = convert_date_columns(loan_data)
loan_data = replace_never_delinquent_value(
    loan_data
)


### Loading Saved Models

In [4]:
import pickle

In [5]:
# PD
with open("../saved_objects/pd_pipeline.pkl", "rb") as f:
    pd_pipeline = pickle.load(f)

# Load scorecard
score_lookup = pd.read_csv(
    "../saved_objects/pd_scorecard.csv"
)

# LGD
with open("../saved_objects/lgd_pipeline.pkl", "rb") as f:
    lgd_pipeline = pickle.load(f)

# EAD
with open("../saved_objects/segment_average_ccf.pkl", "rb") as f:
    segment_ccf = pickle.load(f)
    

### OOT Portfolio

In [6]:
oot_data = loan_data[
    loan_data["dev_oot_flag"] == "OUT_OF_TIME"
].copy()


In [7]:
### Preparing scoring data

oot_scoring = oot_data.copy()


In [8]:
oot_scoring.shape

(249465, 88)

In [9]:
oot_scoring["is_currently_default"].value_counts(dropna=False)

is_currently_default
0    243959
1      5506
Name: count, dtype: int64

In [10]:
### Scoring Population

scoring_data = (
    oot_scoring[
        oot_scoring["is_currently_default"] == 0
    ]
    .copy()
)

print(scoring_data.shape)


(243959, 88)


In [11]:
pd_scoring = scoring_data.copy()

lgd_scoring = scoring_data.copy()

ead_scoring = scoring_data.copy()

### PD Scoring

In [12]:
pd_model = pd_pipeline["model"]

categorical_woe_tables = pd_pipeline["categorical_woe_tables"]

continuous_binning_process = pd_pipeline["continuous_binning_process"]

accepted_categorical = pd_pipeline["accepted_categorical"]

accepted_continuous = pd_pipeline["accepted_continuous"]

variables_to_drop = pd_pipeline["variables_to_drop"]

final_features = pd_pipeline["final_features"]


In [13]:
### PD Preprocessing

# d_times_30dpd_6m_cnt
pd_scoring["d_times_30dpd_6m_cnt"] = (
    pd_scoring["d_times_30dpd_6m_cnt"]
    .astype(str)
    .replace({"4": "4+", "5": "4+", "6": "4+", "7": "4+"})
)

# d_times_30dpd_12m_cnt
pd_scoring["d_times_30dpd_12m_cnt"] = (
    pd_scoring["d_times_30dpd_12m_cnt"]
    .astype(str)
    .replace({"4": "4+", "5": "4+", "6": "4+", "7": "4+"})
)

# d_times_60dpd_12m_cnt
pd_scoring["d_times_60dpd_12m_cnt"] = (
    pd_scoring["d_times_60dpd_12m_cnt"]
    .astype(str)
    .replace({"2": "2+", "3": "2+", "4": "2+"})
)

# delinq_trades_cnt
pd_scoring["delinq_trades_cnt"] = (
    pd_scoring["delinq_trades_cnt"]
    .astype(str)
    .replace({
        "4": "4+",
        "5": "4+",
        "6": "4+",
        "7": "4+",
        "8": "4+",
        "9": "4+",
    })
)

# p_nsf_12m_cnt
pd_scoring["p_nsf_12m_cnt"] = (
    pd_scoring["p_nsf_12m_cnt"]
    .astype(str)
    .replace({"4": "4+", "5": "4+", "6": "4+"})
)

# d_times_90dpd_12m_cnt
pd_scoring["d_times_90dpd_12m_cnt"] = (
    pd_scoring["d_times_90dpd_12m_cnt"]
    .astype(str)
    .replace({"1": "1+", "2": "1+"})
)


In [14]:
# categorical woe transformation
def transform_categorical_to_woe(
    data,
    features,
    woe_tables,
):
    data_woe = pd.DataFrame(index=data.index)

    for feature in features:

        woe_mapping = (
            woe_tables[feature]
            .set_index("category")["woe"]
        )

        data_woe[feature] = (
            data[feature]
            .map(woe_mapping)
            .fillna(0)
        )

    return data_woe
    

In [15]:
categorical_woe = transform_categorical_to_woe(
    pd_scoring,
    accepted_categorical,
    categorical_woe_tables,
)

In [16]:
# continuous woe transformation
continuous_features = continuous_binning_process.variable_names

continuous_woe = continuous_binning_process.transform(
    pd_scoring[continuous_features],
    metric="woe",
)

continuous_woe = continuous_woe[
    accepted_continuous
]


In [17]:
# combining categorical and continuous woe features
pd_woe = pd.concat(
    [
        categorical_woe,
        continuous_woe,
    ],
    axis=1,
)


In [18]:
# dropping the variables
pd_woe = pd_woe.drop(
    columns=variables_to_drop,
    errors="ignore",
)

In [19]:
# aligning feature order
pd_woe = pd_woe[
    final_features
]

In [20]:
# adding intercept
import statsmodels.api as sm

pd_sm = sm.add_constant(
    pd_woe,
    has_constant="add",
)

In [21]:
# predict pd
pd_scoring["pd_pred"] = pd_model.predict(
    pd_sm
)


In [22]:
# checking predictions
pd_scoring["pd_pred"].describe()

count    243959.000000
mean          0.035845
std           0.045983
min           0.002835
25%           0.008990
50%           0.018536
75%           0.041521
max           0.327476
Name: pd_pred, dtype: float64

### LGD Prediction

In [23]:
from src.preprocessing import preprocess_lgd

lgd_model = lgd_pipeline["model"]

X_lgd = preprocess_lgd(
    lgd_scoring,
    lgd_pipeline,
)

lgd_scoring["lgd_pred"] = lgd_model.predict(
    X_lgd
)

lgd_scoring["lgd_pred"].describe()

c:\Users\HP\anaconda3\Lib\site-packages\sklearn\preprocessing\_encoders.py:262: UserWarning: Found unknown categories in columns [8] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


count    243959.000000
mean          0.869182
std           0.049946
min           0.052574
25%           0.854970
50%           0.878963
75%           0.898428
max           0.934019
Name: lgd_pred, dtype: float64

### EAD Scoring

In [24]:
segment_ccf= segment_ccf.reset_index()

ead_scoring = ead_scoring.merge(
    segment_ccf,
    on="portfolio_segment_cd",
    how="left",
)

ead_scoring["ccf"].describe()


count    243959.000000
mean          0.650435
std           0.002010
min           0.649197
25%           0.649197
50%           0.649197
75%           0.653275
max           0.654504
Name: ccf, dtype: float64

In [25]:
ead_scoring["ead_pred"] = (
    ead_scoring["drawn_exposure_amt"]
    + (
        ead_scoring["ccf"]
        * ead_scoring["undrawn_limit_amt"]
    )
)

ead_scoring["ead_pred"].describe()

count    243959.000000
mean       4850.261351
std        3917.651558
min         324.598443
25%        2346.479139
50%        3773.538842
75%        6082.943731
max       82514.734497
Name: ead_pred, dtype: float64

In [26]:
# combining the predictions
ecl_data = (
    pd_scoring[
        ["acct_id", "snapshot_dt", "pd_pred"]
    ]
    .merge(
        lgd_scoring[
            ["acct_id", "snapshot_dt", "lgd_pred"]
        ],
        on=["acct_id", "snapshot_dt"],
        how="inner",
    )
    .merge(
        ead_scoring[
            ["acct_id", "snapshot_dt", "ead_pred"]
        ],
        on=["acct_id", "snapshot_dt"],
        how="inner",
    )
    
)

ecl_data.head()


,acct_id,snapshot_dt,pd_pred,lgd_pred,ead_pred
0,7c58f536e80664ff,2024-07-24,0.224929,0.809259,19401.460292
1,78c8851667dc3967,2024-09-22,0.005198,0.878327,4245.104435
2,1cdc6de16ea1e3e6,2024-12-21,0.030089,0.853344,2777.167975
3,27ad752c1e1f2835,2024-07-24,0.018131,0.876109,3204.501338
4,627023df8625ffbf,2024-07-24,0.143416,0.832310,2347.246345


### ECL

In [27]:
ecl_data["ecl"] = (
    ecl_data["pd_pred"]
    * ecl_data["lgd_pred"]
    * ecl_data["ead_pred"]
)


In [28]:
ecl_data["ecl"].describe()

count    243959.000000
mean        156.796874
std         293.515893
min           0.865462
25%          25.917874
50%          62.619895
75%         160.417514
max       11317.960010
Name: ecl, dtype: float64

In [29]:
portfolio_ecl = ecl_data["ecl"].sum()

print(f"Performing Portfolio Expected Credit Loss: {portfolio_ecl:,.2f}")

Performing Portfolio Expected Credit Loss: 38,252,008.61


In [30]:
average_ecl = ecl_data["ecl"].mean()

print(f"Average ECL per Account (Performing): {average_ecl:,.2f}")

Average ECL per Account (Performing): 156.80


In [31]:
# Top 10 accounts by ECL
ecl_data.sort_values(
    "ecl",
    ascending=False,
).head(10)


,acct_id,snapshot_dt,pd_pred,lgd_pred,ead_pred,ecl
103406,6467691b3acb9d22,2024-10-22,0.159596,0.859440,82514.734497,11317.960010
180653,0d75e2137c3a6a04,2024-11-21,0.220852,0.838287,59037.975953,10930.145074
151441,7f93d7ec650dac10,2024-11-21,0.283833,0.797819,47042.725988,10652.692546
233612,0c3e2a84cd652329,2024-11-21,0.275670,0.787476,36848.610000,7999.237601
35305,629befd81616240c,2024-11-21,0.256464,0.805865,35108.964472,7256.166533
69536,1aa110a04338abbc,2024-07-24,0.275083,0.804636,32431.220784,7178.365225
71614,236c4acbcd3b2900,2024-10-22,0.279660,0.812373,31072.279199,7059.253427
62728,7ffc30ef4c79ce89,2024-11-21,0.201452,0.822330,42291.926558,7006.071000
16026,0d80d0d7fb971f1a,2024-10-22,0.194385,0.832928,42414.451331,6867.289927
93332,0321931655b09d20,2024-09-22,0.257126,0.830912,31920.806190,6819.846099


Defaulted Population Scoring

IFRS 9 requires ECL to be calculated for the FULL portfolio, including accounts already in default (Stage 3). These accounts were excluded from the PD population above (PD is undefined for accounts already defaulted). Here PD = 1 (default has already occurred, it is not a probability), LGD is estimated using the same fitted LGD model applied to their own current characteristics, and EAD uses their actual realized exposure (ead_amt) rather than a CCF-based projection, since it is already known.


In [32]:
defaulted_scoring= oot_scoring[
    oot_scoring["is_currently_default"] == 1
].copy()

print(defaulted_scoring.shape)

(5506, 88)


In [33]:
# PD= 1: already defaulted, not a probability
defaulted_scoring["pd_pred"]= 1.0

In [34]:
# LGD: applying the same fitted beta model to thsoe accounts own current features
X_lgd_defaulted= preprocess_lgd(
    defaulted_scoring,
    lgd_pipeline,
)

defaulted_scoring["lgd_pred"]= lgd_model.predict(
        X_lgd_defaulted
)

defaulted_scoring["lgd_pred"].describe()

count    5506.000000
mean        0.797505
std         0.061255
min         0.297483
25%         0.781424
50%         0.805775
75%         0.828767
max         0.897806
Name: lgd_pred, dtype: float64

In [35]:
# EAD: for already defaulted accounts, exposure at default is already
# known/realized . using their actual ead_amt directly

defaulted_scoring["ead_pred"] = defaulted_scoring["ead_amt"]

defaulted_scoring["ead_pred"].describe()

count     5506.000000
mean      5740.663229
std       4683.589862
min        446.760000
25%       2783.102500
50%       4461.530000
75%       7179.995000
max      57159.410000
Name: ead_pred, dtype: float64

In [36]:
defaulted_ecl = defaulted_scoring[
    ["acct_id", "snapshot_dt", "pd_pred", "lgd_pred", "ead_pred"]
].copy()

In [37]:
defaulted_ecl["ecl"] = (
    defaulted_ecl["pd_pred"]
    * defaulted_ecl["lgd_pred"]
    * defaulted_ecl["ead_pred"]
)

In [38]:
defaulted_ecl["ecl"].describe()

count     5506.000000
mean      4573.333312
std       3758.440914
min        226.806456
25%       2208.618920
50%       3550.745398
75%       5707.293624
max      49052.208657
Name: ecl, dtype: float64

In [39]:
defaulted_portfolio_ecl = defaulted_ecl["ecl"].sum()

print(f"Defaulted Portfolio Expected Credit Loss: {defaulted_portfolio_ecl:,.2f}")

Defaulted Portfolio Expected Credit Loss: 25,180,773.22


### Full Portfolio ECL (Performing + Defaulted)

In [40]:
full_portfolio_ecl = pd.concat(
    [ecl_data, defaulted_ecl],
    ignore_index=True,
)

print(full_portfolio_ecl.shape)

(249465, 6)


In [41]:
portfolio_ecl_total = full_portfolio_ecl["ecl"].sum()

print(f"Full Portfolio Expected Credit Loss: {portfolio_ecl_total:,.2f}")

Full Portfolio Expected Credit Loss: 63,432,781.83


In [42]:
average_ecl_full = full_portfolio_ecl["ecl"].mean()
print(f"Average ECL per Account (Full Portfolio): {average_ecl_full:,.2f}")

Average ECL per Account (Full Portfolio): 254.28


In [43]:
# Top 10 accounts by ECL (full portfolio)
full_portfolio_ecl.sort_values(
    "ecl",
    ascending=False,
).head(10)

,acct_id,snapshot_dt,pd_pred,lgd_pred,ead_pred,ecl
247744,3f278749b458697b,2024-12-21,1.0,0.858165,57159.41,49052.208657
245522,6652808102fdeca4,2024-07-24,1.0,0.803774,51956.19,41761.037690
247671,5744c17e68824c78,2024-12-21,1.0,0.781426,52870.74,41314.553575
248900,3a08c527e1addfdc,2024-12-21,1.0,0.756072,51041.53,38591.051973
245062,749e47bf94d70185,2024-10-22,1.0,0.835355,45155.53,37720.918135
249338,6d17178f7689635d,2024-09-22,1.0,0.824566,45335.42,37382.067233
244522,05ced19cc01318c2,2024-08-23,1.0,0.778228,44577.50,34691.436433
249115,035c07ff7ad05c00,2024-09-22,1.0,0.789463,43519.28,34356.882496
246442,3da32da468383c93,2024-10-22,1.0,0.777511,43063.64,33482.463654
245831,7e85a29cce3f6a2f,2024-08-23,1.0,0.841132,38084.61,32034.202045


In [44]:
# Portfolio summary table
portfolio_summary = pd.DataFrame({
    "Metric": [
        "Accounts Scored (Performing)",
        "Accounts Scored (Defaulted)",
        "Accounts Scored (Total)",
        "Average PD (Full Portfolio)",
        "Average LGD (Full Portfolio)",
        "Average EAD (Full Portfolio)",
        "Average ECL per Account (Full Portfolio)",
        "Portfolio ECL - Performing",
        "Portfolio ECL - Defaulted",
        "Portfolio ECL - Total",
    ],
    "Value": [
        f"{len(ecl_data):,}",
        f"{len(defaulted_ecl):,}",
        f"{len(full_portfolio_ecl):,}",
        f"{full_portfolio_ecl['pd_pred'].mean():.4f}",
        f"{full_portfolio_ecl['lgd_pred'].mean():.4f}",
        f"{full_portfolio_ecl['ead_pred'].mean():,.2f}",
        f"{full_portfolio_ecl['ecl'].mean():,.2f}",
        f"{portfolio_ecl:,.2f}",
        f"{defaulted_portfolio_ecl:,.2f}",
        f"{portfolio_ecl_total:,.2f}",
    ],
})

portfolio_summary


,Metric,Value
0,Accounts Scored (Performing),"243,959"
1,Accounts Scored (Defaulted),"5,506"
2,Accounts Scored (Total),"249,465"
3,Average PD (Full Portfolio),0.0571
4,Average LGD (Full Portfolio),0.8676
5,Average EAD (Full Portfolio),"4,869.91"
6,Average ECL per Account (Full Portfolio),254.28
7,Portfolio ECL - Performing,"38,252,008.61"
8,Portfolio ECL - Defaulted,"25,180,773.22"
9,Portfolio ECL - Total,"63,432,781.83"


### Saving Results

In [45]:
full_portfolio_ecl.to_csv(
    "../data/processed/ecl_predictions.csv",
    index=False,
)
 
portfolio_summary.to_csv(
    "../data/processed/portfolio_summary.csv",
    index=False,
)
 
print("ECL predictions saved successfully.")
print("Portfolio summary saved successfully.")

ECL predictions saved successfully.
Portfolio summary saved successfully.


### Conclusion

This project developed an end-to-end Expected Credit Loss (ECL) framework by integrating three independently developed credit risk models: Probability of Default (PD), Loss Given Default (LGD), and Exposure at Default (EAD). Consistent with IFRS 9 requirements, ECL was calculated across the full Out-of-Time (OOT) portfolio — both performing accounts (Stage 1/2) and accounts already in default (Stage 3) — rather than the performing population alone.

For performing accounts (243,959, ~97.8% of the portfolio), PD was estimated using a WoE-transformed logistic regression scorecard, and LGD/EAD were projected using the same fitted LGD Beta Regression model and segment-average CCF approach, respectively. For already-defaulted accounts (5,506, ~2.2% of the portfolio), PD was set to 1.0 (default has already occurred), LGD was estimated using the same fitted model applied to each account's current characteristics, and EAD used the account's actual realized exposure rather than a projected value.

The individual model outputs were combined using the standard ECL formula: ECL = PD × LGD × EAD.


        	    Performing      	Defaulted       	Total

Accounts	    243,959	              5,506	            249,465

Avg PD	        3.58%	              100%	            5.71%

Avg LGD	        86.92%	              79.75%	        86.76%

Avg EAD	        4,850.26	         5,740.66	        4,869.91

Avg ECL/account	156.80	             4,573.33	        254.28

Portfolio ECL	38,252,008.61	     25,180,773.22	    63,432,781.83

Notably, although already-defaulted accounts represent only 2.2% of the portfolio by count, they account for approximately 39.7% of total portfolio ECL — underscoring the materiality of including Stage 3 accounts in the reserve calculation and confirming why IFRS 9 mandates their inclusion rather than treating ECL as a performing-portfolio-only exercise.

This project demonstrates a complete, IFRS 9-aligned credit risk modeling workflow — from data preparation and independent PD/LGD/EAD model development through full-portfolio ECL aggregation — using reusable preprocessing pipelines and saved model artifacts to separate model development from inference, supporting a scalable, production-oriented approach to reserve calculation.
